# Can we predict next-month churn?
Target: `churn_next_month`. Follows `checklist.md`.

Design decisions (made before any modelling):
- **Leakage:** `lifetime_months` comes from a CRM export of 2026-01-05, i.e. after the fact (it encodes when the account ended). Not known at prediction time, so it is dropped. `account_ref` is an identifier and is dropped; `snapshot_month` is not a feature either (used only for ordering).
- **Split:** accounts appear in many monthly rows and the model is meant for future accounts/months. The test set is the **latest-starting 20% of accounts, with all their rows** (whole entities, later in time). Train CV uses the same structure: forward-chaining over account-start cohorts, whole accounts per fold.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df = pd.read_csv('data.csv')
df['start'] = df.groupby('account_ref').snapshot_month.transform('min')
print(df.shape, df.account_ref.nunique(), df.churn_next_month.mean())

## 1. Hold out the test set (before any modelling)

In [ ]:
acc = df.groupby('account_ref').start.min().sort_values(kind='stable')
n_test = int(round(0.2*len(acc)))
test_acc = set(acc.index[-n_test:])
test = df[df.account_ref.isin(test_acc)].copy()
train = df[~df.account_ref.isin(test_acc)].copy()
assert not set(train.account_ref) & set(test.account_ref)
print(len(train), len(test), 'test accounts start >=', acc.iloc[-n_test], '; train accounts start <=', acc.iloc[-n_test-1])
print('churn rate train/test', train.churn_next_month.mean(), test.churn_next_month.mean())

## 2. Cross-validation structure (training rows only)
Train accounts are ordered by start month and cut into 6 cohorts. Forward chaining: fold k trains on accounts in cohorts < k and validates on the whole accounts of cohort k (k = 1..5).

In [ ]:
tacc = train.groupby('account_ref').start.min().sort_values(kind='stable')
cohort_of = {}
for k, idx in enumerate(np.array_split(np.arange(len(tacc)), 6)):
    for a in tacc.index[idx]: cohort_of[a] = k
train['cohort'] = train.account_ref.map(cohort_of)
folds = [(np.where(train.cohort<k)[0], np.where(train.cohort==k)[0]) for k in range(1,6)]
for tr,va in folds: print(len(tr), len(va), train.iloc[va].churn_next_month.mean().round(3))

## 3. Candidate models (named before scoring)
Features known at prediction time: `plan, seats, monthly_fee, tenure_months, usage_hours, support_tickets`.
1. `baseline`: predicts training prevalence (trivial).
2. `logreg`: logistic regression, C in {0.1, 1, 10}.
3. `hgb`: HistGradientBoosting, max_depth in {3, None}, lr 0.05.
4. `rf`: random forest, min_samples_leaf in {5, 20}.

Selection metric: mean CV average precision (~15% positives). The decision threshold is chosen by CV F1 on pooled out-of-fold predictions. All preprocessing sits inside the pipeline, so it is fit per fold.

In [ ]:
num=['seats','monthly_fee','tenure_months','usage_hours','support_tickets']; cat=['plan']
feats=num+cat
def pre(scale): return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
cands={'baseline':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]: cands[f'logreg_C{C_}']=Pipeline([('p',pre(True)),('m',LogisticRegression(C=C_,max_iter=2000))])
for d in [3,None]: cands[f'hgb_d{d}']=Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=0))])
for l in [5,20]: cands[f'rf_leaf{l}']=Pipeline([('p',pre(False)),('m',RandomForestClassifier(300,min_samples_leaf=l,n_jobs=-1,random_state=0))])
X,y=train[feats],train.churn_next_month.values
oof={}; rows=[]
for name,m in cands.items():
    p=np.full(len(train),np.nan); ap=[];auc=[]
    for tr,va in folds:
        f=clone(m).fit(X.iloc[tr],y[tr]); p[va]=f.predict_proba(X.iloc[va])[:,1]
        ap.append(average_precision_score(y[va],p[va])); auc.append(roc_auc_score(y[va],p[va]))
    oof[name]=p; rows.append((name,np.mean(ap),np.std(ap),np.mean(auc)))
cv=pd.DataFrame(rows,columns=['model','cv_AP','AP_sd','cv_AUC']).sort_values('cv_AP',ascending=False); cv

## 4. Choose model and threshold on CV only

In [ ]:
best=cv.iloc[0].model
mask=~np.isnan(oof[best])
ths=np.linspace(0.05,0.8,76)
f1s=[f1_score(y[mask],oof[best][mask]>t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(thr,3),'| CV F1:',round(max(f1s),3))
print('baseline CV AP (= prevalence):', cv.set_index('model').loc['baseline','cv_AP'])

**Choice recorded before test scoring:** the model with the highest mean CV average precision (printed above) and the F1-maximising out-of-fold threshold. No changes after seeing the test set.

## 5. Refit on all training rows; score the test set once (headline)

In [ ]:
final=clone(cands[best]).fit(X,y)
pt=final.predict_proba(test[feats])[:,1]; yt=test.churn_next_month.values
base=clone(cands['baseline']).fit(X,y).predict_proba(test[feats])[:,1]
res=dict(AUC=roc_auc_score(yt,pt),AP=average_precision_score(yt,pt),prevalence=yt.mean(),
         F1_at_thr=f1_score(yt,pt>thr),Brier=brier_score_loss(yt,pt),Brier_baseline=brier_score_loss(yt,base))
# account-level bootstrap of the fixed predictions, for uncertainty only (no refitting)
rng=np.random.default_rng(0); g=test.account_ref.values; ids=np.unique(g); idx={a:np.where(g==a)[0] for a in ids}
bs=[]
for _ in range(500):
    ii=np.concatenate([idx[a] for a in rng.choice(ids,len(ids))]); bs.append((roc_auc_score(yt[ii],pt[ii]),average_precision_score(yt[ii],pt[ii])))
lo=np.percentile(bs,[2.5,97.5],axis=0)
print({k:round(v,3) for k,v in res.items()}); print('AUC 95% CI',lo[:,0].round(3),'AP 95% CI',lo[:,1].round(3))

## Conclusion
The headline is the single test-set score above for the pre-chosen model, compared with the prevalence baseline (AUC 0.5, AP = prevalence). `lifetime_months` was excluded as leakage. Caveats: the test set is one later-starting cohort of accounts, rows within an account are correlated (hence the account-level bootstrap), and the threshold is optimised for F1, not business cost.